<a href="https://colab.research.google.com/github/detlingmatheus7-oss/detlingmatheu7-oss-YOLOv12-Diatraea-Saccharalis-Egg-Detection/blob/main/Train_YOLO12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLOv12

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
!pip install --q ultralytics

In [ ]:
import gc
import torch

del model

gc.collect()
torch.cuda.empty_cache()

In [ ]:
 from ultralytics import YOLO

 # n ,s, m, l, x
 model = YOLO('yolo12x.pt')

 config = '/content/drive/MyDrive/project/config.yaml'

 model.train(
             data=config,
             epochs=150,

             imgsz=1280,
             batch=4,

             optimizer='AdamW',
             lr0=0.0005,

             mosaic=1.0,
             mixup=0.1,

             scale=0.5, #Original

             flipud=0.1,
             fliplr=0.5,

             cache=True,
             freeze=8,
             box=8.0
             )

In [ ]:
#Importa a biblioteca shutil para que possamos salvar o que fizemos do Colab para o Drive
import shutil

#Defina os diretórios de origem e destino
diretorio_origem = '/content/runs/detect/train'
diretorio_destino = '/content/drive/MyDrive/project/Modelo'

#Copie os arquivos recursivamente
shutil.copytree(diretorio_origem, diretorio_destino)

In [ ]:
import os
import cv2
import shutil
import random
import albumentations as A

input_img_folder = '/content/drive/MyDrive/project/images/all'
input_label_folder = '/content/drive/MyDrive/project/labels/all'

output_img_folder = '/content/drive/MyDrive/projectA/images/all'
output_label_folder = '/content/drive/MyDrive/projectA/labels/all'

os.makedirs(output_img_folder, exist_ok=True)
os.makedirs(output_label_folder, exist_ok=True)

# Apenas transformações que não mexem na geometria
transform = A.Compose([
    A.RandomBrightnessContrast(p=0.4),
    A.HueSaturationValue(p=0.25),
    A.GaussNoise(p=0.00),
    A.GaussianBlur(p=0.00),
])

for filename in os.listdir(input_img_folder):

    if filename.endswith(".png"):

        # Apenas 30% das imagens serão aumentadas
        if random.random() > 0.40:
            continue

        img_path = os.path.join(input_img_folder, filename)
        label_name = filename.replace(".png", ".txt")

        label_path = os.path.join(input_label_folder, label_name)

        image = cv2.imread(img_path)

        augmented = transform(image=image)
        aug_image = augmented["image"]

        new_img_name = filename.replace(".png", "_aug.png")
        new_label_name = label_name.replace(".txt", "_aug.txt")

        cv2.imwrite(os.path.join(output_img_folder, new_img_name), aug_image)

        if os.path.exists(label_path):
            shutil.copy(
                label_path,
                os.path.join(output_label_folder, new_label_name)
            )
        else:
            print(f"⚠️ Label não encontrada: {label_name}")

print("Augmentação + cópia de labels concluída 🚀")

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/project/Modelo/weights/best.pt')

image_folder = '/content/drive/MyDrive/project/images/test'

results = model.predict(source=image_folder,save=True, save_txt=True, conf=0.5)

In [ ]:
import pandas as pd
import re

# Cole aqui o log bruto (copie e cole todo aquele texto do terminal)
raw_log = """
"""

def parse_yolo_log(log_text):
    data = []
    # Regex para capturar o nome do arquivo e a quantidade de ovos
    pattern = re.compile(r"test/(.*?\.png):.*?(\d+) eggss")

    for line in log_text.strip().split('\n'):
        match = pattern.search(line)
        if match:
            filename = match.group(1)
            count = int(match.group(2))

            # Filtra apenas as imagens que NÃO possuem '_aug' no nome
            if "_aug" not in filename:
                data.append({"Imagem": filename, "Ovos Detectados": count})

    return pd.DataFrame(data)

# Processar e exibir
df_resultados = parse_yolo_log(raw_log)

# Exibe a tabela formatada
print(df_resultados.to_string(index=False))

# Opcional: Calcular o total identificado pelo modelo nas originais
print(f"\nTotal acumulado nas imagens: {df_resultados['Ovos Detectados'].sum()}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# Caminho da pasta contendo as imagens
folder_path = '/content/runs/detect/predict'

# Listar todos os arquivos na pasta
files = os.listdir(folder_path)

# Filtrar apenas os arquivos de imagem (opcional)
image_files = [f for f in files if f.endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.JPG'))]

# Loop para carregar e exibir cada imagem
for image_file in image_files:
    img_path = os.path.join(folder_path, image_file)
    image = mpimg.imread(img_path)
    plt.imshow(image)
    plt.axis('off')
    plt.show()